# 📄 Text Information Extractor
Extracts **Name**, **Email**, **Phone Number**, **Skills**, **Education**, and **Experience** from any raw text.

> ✅ **Zero external dependencies** — uses only Python's built-in `re` and `json` modules.

In [2]:
import re
import json

print('✅ Libraries loaded — no DLL dependencies!')

✅ Libraries loaded — no DLL dependencies!


In [3]:
# ─────────────────────────────────────────────
# 🔧 Extraction Functions
# ─────────────────────────────────────────────

# ── Email ──────────────────────────────────────
def extract_email(text):
    """Extract first email address found in the text."""
    pattern = r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}'
    match = re.search(pattern, text)
    return match.group() if match else None


# ── Phone ──────────────────────────────────────
def extract_phone(text):
    """Extract first phone number (handles +91, dashes, dots, spaces)."""
    pattern = r'(\+?\d[\d\s\-().]{7,15}\d)'
    matches = re.findall(pattern, text)
    for m in matches:
        digits_only = re.sub(r'\D', '', m)
        if 7 <= len(digits_only) <= 15:
            return m.strip()
    return None


# ── Name ───────────────────────────────────────
# Strategy: the name is usually on one of the first non-empty lines,
# before any label like 'email:', 'phone:', etc.
def extract_name(text):
    """Heuristic: first short non-label line that looks like a proper name."""
    lines = text.strip().split('\n')
    SKIP_LABELS = re.compile(
        r'^(email|phone|mobile|address|linkedin|github|website|summary|objective|name\s*:)',
        re.IGNORECASE
    )
    NAME_PATTERN = re.compile(r'^[A-Z][a-z]+(\s[A-Z][a-z]+){1,3}$')
    for line in lines[:10]:           # check first 10 lines only
        stripped = line.strip()
        if not stripped:
            continue
        if SKIP_LABELS.match(stripped):
            # Try extracting the value after the colon
            if ':' in stripped:
                val = stripped.split(':', 1)[1].strip()
                if NAME_PATTERN.match(val):
                    return val
            continue
        if NAME_PATTERN.match(stripped):
            return stripped
    return None


# ── Skills ─────────────────────────────────────
SKILLS_DB = [
    # Languages
    'python', 'java', 'javascript', 'typescript', 'c++', 'c#', 'c', 'ruby',
    'go', 'rust', 'swift', 'kotlin', 'php', 'r', 'scala', 'perl', 'matlab',
    # Web
    'html', 'css', 'react', 'angular', 'vue', 'node.js', 'nodejs', 'django',
    'flask', 'fastapi', 'spring', 'express', 'next.js', 'nuxt',
    # Data & ML
    'machine learning', 'deep learning', 'nlp', 'computer vision',
    'tensorflow', 'pytorch', 'keras', 'scikit-learn', 'pandas', 'numpy',
    'matplotlib', 'seaborn', 'opencv', 'hugging face', 'transformers',
    # Databases
    'sql', 'mysql', 'postgresql', 'mongodb', 'sqlite', 'redis',
    'elasticsearch', 'firebase', 'dynamodb', 'oracle',
    # Cloud & DevOps
    'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'terraform', 'ansible',
    'jenkins', 'ci/cd', 'git', 'github', 'gitlab', 'linux', 'bash',
    # General
    'agile', 'scrum', 'rest api', 'graphql', 'microservices',
    'data analysis', 'data science', 'excel', 'power bi', 'tableau',
    'spark', 'hadoop',
]

def extract_skills(text):
    """Match known skills from text (case-insensitive)."""
    text_lower = text.lower()
    found = [
        skill for skill in SKILLS_DB
        if re.search(r'(?<![a-z])' + re.escape(skill) + r'(?![a-z])', text_lower)
    ]
    return list(dict.fromkeys(found))   # preserve order, drop duplicates


# ── Education ──────────────────────────────────
EDUCATION_KEYWORDS = [
    'b.tech', 'b.e', 'b.sc', 'b.com', 'b.a', 'bca', 'bba',
    'm.tech', 'm.e', 'm.sc', 'm.com', 'm.a', 'mca', 'mba',
    'phd', 'ph.d', 'bachelor', 'master', 'doctorate',
    'high school', 'secondary', 'diploma', '12th', '10th',
    'university', 'college', 'institute',
]

SECTION_STOP = re.compile(
    r'^(experience|skills|projects|certifications|achievements|languages|publications)',
    re.IGNORECASE
)

def extract_education(text):
    """Grab lines under the Education section or containing education keywords."""
    lines = text.split('\n')
    results, in_section = [], False

    for line in lines:
        s = line.strip()
        if not s:
            continue
        if re.search(r'\b(education|academic|qualification)\b', s, re.IGNORECASE):
            in_section = True
            continue
        if in_section and SECTION_STOP.match(s):
            break
        if in_section and len(s) > 4:
            results.append(s)
        elif any(kw in s.lower() for kw in EDUCATION_KEYWORDS) and s not in results:
            results.append(s)

    return results if results else None


# ── Experience ─────────────────────────────────
EXPERIENCE_STOP = re.compile(
    r'^(education|skills|projects|certifications|achievements|languages|publications)',
    re.IGNORECASE
)

def extract_experience(text):
    """Grab lines under the Experience / Work History section."""
    lines = text.split('\n')
    results, in_section = [], False

    for line in lines:
        s = line.strip()
        if not s:
            continue
        if re.search(r'\b(experience|work history|employment|professional experience)\b', s, re.IGNORECASE):
            in_section = True
            continue
        if in_section and EXPERIENCE_STOP.match(s):
            break
        if in_section and len(s) > 4:
            results.append(s)

    return results if results else None


# ── Master function ─────────────────────────────
def extract_all(text):
    """Run all extractors and return a structured dictionary."""
    return {
        'Name'       : extract_name(text),
        'Email'      : extract_email(text),
        'Phone'      : extract_phone(text),
        'Skills'     : extract_skills(text),
        'Education'  : extract_education(text),
        'Experience' : extract_experience(text),
    }

print('✅ All extraction functions defined!')

✅ All extraction functions defined!


In [8]:
# ─────────────────────────────────────────────
# 📋 Sample Input Text
# ─────────────────────────────────────────────

sample_text = """
Vansh Parmar
§ Github | ï Linkedin |  Leetcode | # vanshpar6@gmail.com | H +91 8799630700
Projects
Uber Backend Project Github Link
Developed a scalable Spring Boot microservices platform with Eureka for service discovery and real-time
driver–client tracking using Redis and WebSocket.
Implemented JWT authentication and inter-service communication with Retrofit, and built modular booking, review, and Auth services using Spring Security, JPA, and REST APIs.
Document Q&A Chatbot Demo Link
Built a RAG-based system using LangChain, FAISS, Groq LLMs and HuggingFace embeddings to enable
accurate, context-aware question answering over PDF documents. Designed a clean UI using Streamlit
Electricity Demand Prediction Github Link
Trained an XGBoost Regressor on five years of historical data, engineering complex temporal and weather
features while using a chronological split to prevent data leakage. Deployed a real-time Streamlit predictive
web application that forecasts electricity demand with a highly accurate 2.05% average percentage error.
App Builder - AI Agent Github Link
Developed an AI-powered application builder agent using LangGraph, LangChain, and Groq LLMs, enabling end-to-end project generation from natural language inputs. Implemented planner, architect, and
coder workflows to automate structured application development.
Education
2023 - 2027 Bachelor’s Engineering in CSE at The Maharaja Sayajirao University
Skills
Languages C++, Java, JavaScript, SQL, Python, HTML5
Fundamentals OOPs, DSA, DBMS, Computer Network, Operating System
Technologies/Frameworks CSS, Spring Boot, Spring Microservices, Spring Security, NumPy, Pandas, Machine
Learning, Deep Learning, OpenCV, NLP, GenAI, RAG, Agentic AI

"""

results = extract_all(sample_text)

print('=' * 55)
print('           EXTRACTED INFORMATION')
print('=' * 55)
for key, value in results.items():
    print(f'\n🔹 {key}:')
    if isinstance(value, list):
        for item in value:
            print(f'   • {item}')
    else:
        print(f'   {value}')
print('\n' + '=' * 55)

           EXTRACTED INFORMATION

🔹 Name:
   Vansh Parmar

🔹 Email:
   vanshpar6@gmail.com

🔹 Phone:
   +91 8799630700

🔹 Skills:
   • python
   • java
   • javascript
   • c++
   • c
   • html
   • css
   • spring
   • deep learning
   • nlp
   • pandas
   • numpy
   • opencv
   • sql
   • redis
   • github
   • microservices

🔹 Education:
   • 2023 - 2027 Bachelor’s Engineering in CSE at The Maharaja Sayajirao University

🔹 Experience:
   None



In [9]:
# ─────────────────────────────────────────────
# 💾 Export Results to JSON
# ─────────────────────────────────────────────

output_path = 'extracted_info.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=4)

print(f'✅ Results saved to {output_path}')

✅ Results saved to extracted_info.json
